<a href="https://colab.research.google.com/github/aapaul29/Distance-Recognition-SML1-/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SML Project 1
This notebook works in both **VS Code (local)** and **Google Colab**.

## 1. Environment Detection & Setup

In [1]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules
print(f"Running in {'Google Colab' if IN_COLAB else 'VS Code / local'}")

Running in VS Code / local


## 2. Google Colab: Mount Drive & Navigate to Project
This cell only runs on Colab. Adjust `COLAB_PROJECT_PATH` to where you uploaded the project folder in your Google Drive.

In [2]:
# ---------------------------------------------------------------
# COLAB ONLY — skip if running locally
# Adjust this path to match your Google Drive folder structure.
COLAB_PROJECT_PATH = "/content/drive/MyDrive/Aaryan/Uni/BS Mech Eng ETH Zurich/2nd year/Semester 4/Stochastics & Machine Learning/Distance-Recognition-SML1-"
# ---------------------------------------------------------------

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd $COLAB_PROJECT_PATH
    print(f"Working directory: {os.getcwd()}")

    # Unzip data if not already extracted
    data_zip = os.path.join(COLAB_PROJECT_PATH, "data.zip")
    data_dir = os.path.join(COLAB_PROJECT_PATH, "data")
    if os.path.exists(data_zip) and not os.path.exists(data_dir):
        print("Unzipping data...")
        os.makedirs(data_dir, exist_ok=True)
        !unzip -q $data_zip -d $data_dir
        print("Done.")
    else:
        print("Data directory already exists or no zip found — skipping unzip.")
else:
    print("Local environment — no mounting needed.")

Local environment — no mounting needed.


## 3. Install / Verify Dependencies

In [3]:
# On Colab, install any missing packages from requirements.txt.
# Locally, this is a no-op if your conda env is already set up.
if IN_COLAB:
    !pip install -q -r requirements.txt

## 4. Auto-reload & Imports

In [4]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from utils import load_config, load_dataset, load_test_dataset, print_results, save_results

# Add further sklearn imports here as needed.
# Note: SVRs are NOT allowed in this project.

## 5. Load Config & Data

In [5]:
config = load_config()

images, distances = load_dataset(config, split="train")
print(f"[INFO]: Dataset loaded with {len(images)} samples.")
print(f"[INFO]: Feature dimensionality: {images.shape[1]}")

[INFO]: Configs are loaded with: 
 {'data_dir': PosixPath('data'), 'load_rgb': True, 'downsample_factor': 3}
[INFO]: Dataset loaded with 3000 samples.
[INFO]: Feature dimensionality: 30000


## 6. Your Implementation
Implement preprocessing, model training, and evaluation below.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor, RadiusNeighborsRegressor, NearestNeighbors
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.base import BaseEstimator, RegressorMixin
import numpy as np, time

ds = config["downsample_factor"]
H = W = 300 // ds
C = 3 if config["load_rgb"] else 1
print(f"Image size: {H}x{W}, channels: {C}")

# ===== Helper functions =====
def numpy_sobel(img, axis):
    if axis == 0:
        k = np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=np.float32)
    else:
        k = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=np.float32)
    out = np.zeros_like(img, dtype=np.float32)
    for di in range(3):
        for dj in range(3):
            out[1:-1,1:-1] += k[di,dj] * img[di:di+img.shape[0]-2, dj:dj+img.shape[1]-2]
    return out

def to_gray(img_flat):
    if C == 3:
        return img_flat.reshape(H, W, 3).mean(axis=2).astype(np.float32)
    return img_flat.reshape(H, W).astype(np.float32)

def extract_lbp(img_2d):
    lbp = np.zeros_like(img_2d, dtype=np.uint8)
    center = img_2d[1:-1, 1:-1]
    for i, n in enumerate([
        img_2d[0:-2,0:-2], img_2d[0:-2,1:-1], img_2d[0:-2,2:],
        img_2d[1:-1,2:], img_2d[2:,2:], img_2d[2:,1:-1],
        img_2d[2:,0:-2], img_2d[1:-1,0:-2]
    ]):
        lbp[1:-1,1:-1] += (n >= center).astype(np.uint8) * (2**i)
    hist, _ = np.histogram(lbp, bins=32, range=(0,256))
    return hist.astype(np.float32) / (hist.sum() + 1e-8)

# ===== Model classes =====
class HGBClf(BaseEstimator, RegressorMixin):
    def __init__(self, n_bins=30, **kwargs):
        self.n_bins = n_bins
        self.kwargs = kwargs
    def fit(self, X, y):
        edges = np.linspace(y.min()-0.01, y.max()+0.01, self.n_bins+1)
        self.centers_ = (edges[:-1] + edges[1:]) / 2
        yb = np.clip(np.digitize(y, edges)-1, 0, self.n_bins-1)
        self.clf_ = HistGradientBoostingClassifier(**self.kwargs)
        self.clf_.fit(X, yb)
        self.cc_ = self.centers_[self.clf_.classes_]
        return self
    def predict(self, X):
        return self.clf_.predict_proba(X) @ self.cc_

class SafeRadiusNN(BaseEstimator, RegressorMixin):
    def __init__(self, radius=5.9, weights='distance', fallback_k=2):
        self.radius = radius
        self.weights = weights
        self.fallback_k = fallback_k
    def fit(self, X, y):
        self.rnn_ = RadiusNeighborsRegressor(radius=self.radius, weights=self.weights)
        self.rnn_.fit(X, y)
        self.knn_ = KNeighborsRegressor(n_neighbors=self.fallback_k, weights=self.weights)
        self.knn_.fit(X, y)
        self.X_train_ = X
        return self
    def predict(self, X):
        nn = NearestNeighbors(radius=self.radius)
        nn.fit(self.X_train_)
        dists, _ = nn.radius_neighbors(X)
        has_nb = np.array([len(d) > 0 for d in dists])
        pred = self.knn_.predict(X)
        if has_nb.any():
            try: pred[has_nb] = self.rnn_.predict(X[has_nb])
            except: pass
        return pred

# ===== Feature extraction =====
def extract_knn_features(X_raw, H, W, C):
    """515 features — used by KNN and RadiusNN."""
    all_feat = []
    for img in X_raw:
        gray = to_gray(img)
        wt = np.linspace(0.3, 1.0, H).reshape(-1, 1)
        img_w = gray * wt
        feats = []
        for g in [4, 5]:
            gh, gw = H // g, W // g
            for i in range(g):
                for j in range(g):
                    c = img_w[i*gh:(i+1)*gh, j*gw:(j+1)*gw]
                    feats.extend([c.mean(), c.var()])
        n_strips = 25; sh = H // n_strips; means = []
        for i in range(n_strips):
            s = img_w[i*sh:(i+1)*sh, :]; m = s.mean(); means.append(m)
            feats.extend([m, s.var(), s.min(), s.max()])
        for i in range(1, n_strips): feats.append(means[i] - means[i-1])
        row_means = gray.mean(axis=1); vg = np.diff(row_means)
        feats.extend([vg.mean(), vg.std(), vg.min(), vg.max()])
        feats.append(np.argmax(np.abs(vg)) / H); feats.append(np.argmin(vg) / H)
        feats.append(np.argmax(row_means) / H); feats.append(np.argmin(row_means) / H)
        for p in [25, 50, 75]: feats.append(np.percentile(row_means, p))
        feats.append(row_means[:H//3].mean() - row_means[2*H//3:].mean())
        bt = 2*H//3; bot = img_w[bt:, :]; bh = (H - bt) // 4; bw = W // 4
        if bh > 0:
            for i in range(4):
                for j in range(4):
                    c = bot[i*bh:(i+1)*bh, j*bw:(j+1)*bw]; feats.extend([c.mean(), c.var()])
        hg, _ = np.histogram(gray, bins=16, range=(0,255))
        feats.extend((hg/(hg.sum()+1e-8)).tolist())
        feats.extend([gray.mean(), gray.std(), gray.var()])
        for p in [5,10,25,50,75,90,95]: feats.append(np.percentile(gray, p))
        feats.append(gray[H//2:].mean() / (gray[:H//2].mean()+1e-8))
        feats.append(gray[:H//3].mean() / (gray[2*H//3:].mean()+1e-8))
        feats.append(gray[2*H//3:].std() / (gray[:H//3].std()+1e-8))
        feats.extend(extract_lbp(gray).tolist())
        sx = numpy_sobel(img_w, axis=1); sy = numpy_sobel(img_w, axis=0)
        feats.extend([sx.mean(), sx.var(), sy.mean(), sy.var()])
        hsx, _ = np.histogram(sx, bins=16); hsy, _ = np.histogram(sy, bins=16)
        feats.extend((hsx/(hsx.sum()+1e-8)).tolist()); feats.extend((hsy/(hsy.sum()+1e-8)).tolist())
        grad = np.abs(np.diff(gray, axis=0))
        for t in [3,5,8,10,15,20,25,35]:
            rows = np.where(grad > t, np.arange(H-1).reshape(-1,1), -1).max(axis=0)
            v = np.where(rows >= 0, rows / H, 0).astype(np.float32); valid = v[v > 0]
            if len(valid) > 0:
                feats.extend([np.median(valid), np.mean(valid), np.std(valid),
                              np.percentile(valid,25), np.percentile(valid,75)])
            else: feats.extend([0,0,0,0,0])
        mag = np.sqrt(sx**2 + sy**2); zh = H // 10
        for i in range(10):
            zone = mag[i*zh:(i+1)*zh]; feats.extend([zone.mean(), zone.var()])
        col_means = gray.mean(axis=0); cg = np.diff(col_means)
        feats.extend([col_means.mean(), col_means.std(), cg.mean(), cg.std()])
        feats.append(gray[:,:W//2].mean() / (gray[:,W//2:].mean()+1e-8))
        for p in [10,25,50,75,90]: feats.append(np.percentile(np.min(gray, axis=0), p))
        for p in [10,25,50,75,90]: feats.append(np.percentile(gray[H//2:].mean(axis=0), p))
        if C == 3:
            img_rgb = img.reshape(H,W,3).astype(np.float32)
            r,g,b = img_rgb[:,:,0], img_rgb[:,:,1], img_rgb[:,:,2]
            for ch in [r,g,b]: feats.extend([ch.mean(), ch.std()])
            feats.extend([r.mean()/(g.mean()+1e-8), r.mean()/(b.mean()+1e-8), g.mean()/(b.mean()+1e-8)])
            for ch in [r,g,b]: feats.append(ch[H//2:].mean()/(ch[:H//2].mean()+1e-8))
            for ch in [r,g,b]:
                hc, _ = np.histogram(ch, bins=16, range=(0,255))
                feats.extend((hc/(hc.sum()+1e-8)).tolist())
            sh10 = H // 10
            for ch in [r,g,b]:
                for i in range(10): feats.append(ch[i*sh10:(i+1)*sh10].mean())
            for ch in [r,g,b]: feats.append(ch[2*H//3:].mean() / (ch[:H//3].mean()+1e-8))
        all_feat.append(np.array(feats, dtype=np.float32))
    return np.array(all_feat)

def extract_hgb_extra(X_raw, H, W, C):
    """391 extra features — only for HGB."""
    all_extra = []
    for img in X_raw:
        gray = to_gray(img); wt = np.linspace(0.3, 1.0, H).reshape(-1, 1); img_w = gray * wt
        feats = []
        for g in [6, 8]:
            gh, gw = H // g, W // g
            for i in range(g):
                for j in range(g):
                    c = img_w[i*gh:(i+1)*gh, j*gw:(j+1)*gw]; feats.extend([c.mean(), c.var()])
        sh = H // 25; means = [img_w[i*sh:(i+1)*sh, :].mean() for i in range(25)]
        for i in range(2, 25): feats.append(means[i] - 2*means[i-1] + means[i-2])
        hg32, _ = np.histogram(gray, bins=32, range=(0,255)); feats.extend((hg32/(hg32.sum()+1e-8)).tolist())
        bt = 2*H//3; bot = img_w[bt:, :]; bh_ = (H-bt)//6; bw_ = W//6
        if bh_ > 0 and bw_ > 0:
            for i in range(6):
                for j in range(6):
                    c = bot[i*bh_:(i+1)*bh_, j*bw_:(j+1)*bw_]; feats.extend([c.mean(), c.var()])
        sx = numpy_sobel(img_w, axis=1); sy = numpy_sobel(img_w, axis=0); mag = np.sqrt(sx**2 + sy**2)
        hmag, _ = np.histogram(mag, bins=16); feats.extend((hmag/(hmag.sum()+1e-8)).tolist())
        feats.extend([mag.mean(), mag.var(), mag.max()])
        row_means = gray.mean(axis=1)
        feats.append(row_means[:H//4].mean() - row_means[3*H//4:].mean())
        feats.append(np.sum(row_means * np.arange(H)) / (np.sum(row_means) + 1e-8) / H)
        for p in [10, 90]: feats.append(np.percentile(row_means, p))
        if C == 3:
            img_rgb = img.reshape(H,W,3).astype(np.float32)
            r,g,b = img_rgb[:,:,0], img_rgb[:,:,1], img_rgb[:,:,2]
            for ch in [r,g,b]: feats.append(ch[3*H//4:].mean()/(ch[:H//4].mean()+1e-8)); feats.append(ch.var())
            feats.extend([r[2*H//3:].mean()-g[2*H//3:].mean(), r[2*H//3:].mean()-b[2*H//3:].mean(), g[2*H//3:].mean()-b[2*H//3:].mean()])
        feats.extend(extract_lbp(gray[::2, ::2]).tolist())
        all_extra.append(np.array(feats, dtype=np.float32))
    return np.array(all_extra)

# ===== Split + extract =====
X_train_raw, X_val_raw, y_train, y_val = train_test_split(images, distances, test_size=0.15, random_state=42)
t0 = time.time()
X_tr_knn = extract_knn_features(X_train_raw, H, W, C)
X_va_knn = extract_knn_features(X_val_raw, H, W, C)
X_tr_extra = extract_hgb_extra(X_train_raw, H, W, C)
X_va_extra = extract_hgb_extra(X_val_raw, H, W, C)
print(f"Features extracted ({time.time()-t0:.1f}s): KNN={X_tr_knn.shape[1]}, HGB extra={X_tr_extra.shape[1]}")

# Scale
sc_knn = StandardScaler(); X_tr_s = sc_knn.fit_transform(X_tr_knn); X_va_s = sc_knn.transform(X_va_knn)
X_tr_hgb = np.hstack([X_tr_knn, X_tr_extra]); X_va_hgb = np.hstack([X_va_knn, X_va_extra])
sc_hgb = StandardScaler(); X_tr_hgb_s = sc_hgb.fit_transform(X_tr_hgb); X_va_hgb_s = sc_hgb.transform(X_va_hgb)
print(f"HGB total features: {X_tr_hgb.shape[1]}")

Image size: 100x100, channels: 3
Features extracted (17.6s): KNN=515, HGB extra=391
HGB total features: 906


In [7]:
# ===== Train all models + optimize blend =====
# KNN
knn1 = KNeighborsRegressor(n_neighbors=1); knn1.fit(X_tr_s, y_train)
knn2 = KNeighborsRegressor(n_neighbors=2, weights='distance'); knn2.fit(X_tr_s, y_train)
knn3 = KNeighborsRegressor(n_neighbors=3, weights='distance'); knn3.fit(X_tr_s, y_train)

# RadiusNN r=5.9
rnn = SafeRadiusNN(radius=5.9, weights='distance', fallback_k=2); rnn.fit(X_tr_s, y_train)

# HGB 5-seed on 906 features
print("Training HGB (5 seeds)...")
hgb_seeds = [42, 123, 7, 2024, 999]
hgb_preds_val = []
for seed in hgb_seeds:
    c = HGBClf(n_bins=15, max_iter=500, max_depth=5, learning_rate=0.05,
               min_samples_leaf=15, l2_regularization=0.2, random_state=seed)
    c.fit(X_tr_hgb_s, y_train)
    hgb_preds_val.append(c.predict(X_va_hgb_s))

# MLP α=1.0
print("Training MLP (α=1.0)...")
mlp = MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=1000, early_stopping=True,
                   validation_fraction=0.15, learning_rate='adaptive', learning_rate_init=0.001,
                   alpha=1.0, random_state=42)
mlp.fit(X_tr_s, y_train)

# Predictions
p_knn2 = knn2.predict(X_va_s)
p_knn13 = 0.5 * knn1.predict(X_va_s) + 0.5 * knn3.predict(X_va_s)
p_rnn = rnn.predict(X_va_s)
p_hgb = np.mean(hgb_preds_val, axis=0)
p_mlp = mlp.predict(X_va_s)

print(f"\n--- Individual MAEs ---")
print(f"KNN k=2:      {mean_absolute_error(y_val, p_knn2)*100:.2f} cm")
print(f"KNN(1+3)/2:   {mean_absolute_error(y_val, p_knn13)*100:.2f} cm")
print(f"RadiusNN 5.9: {mean_absolute_error(y_val, p_rnn)*100:.2f} cm")
print(f"HGB 5-seed:   {mean_absolute_error(y_val, p_hgb)*100:.2f} cm")
print(f"MLP α=1.0:    {mean_absolute_error(y_val, p_mlp)*100:.2f} cm")

# Optimize 5-model blend
print("\nOptimizing blend...")
preds = [p_knn2, p_knn13, p_rnn, p_hgb, p_mlp]
names5 = ["KNN2", "KNN13", "RadNN", "HGB5s", "MLP"]
best_mae = 999; best_w = None
for w0 in np.arange(0, 0.5, 0.05):
    for w1 in np.arange(0, 0.6, 0.05):
        for w2 in np.arange(0, 0.4, 0.05):
            for w3 in np.arange(0, 0.5, 0.05):
                w4 = 1.0 - w0 - w1 - w2 - w3
                if w4 < -0.01 or w4 > 0.3: continue
                b = sum(w*p for w,p in zip([w0,w1,w2,w3,max(0,w4)], preds))
                m = mean_absolute_error(y_val, b)*100
                if m < best_mae: best_mae = m; best_w = [w0,w1,w2,w3,max(0,w4)]

np.random.seed(0)
for _ in range(20000):
    ws = np.array(best_w) + np.random.randn(5)*0.02
    ws = np.maximum(ws, 0); ws /= ws.sum()
    m = mean_absolute_error(y_val, sum(w*p for w,p in zip(ws, preds)))*100
    if m < best_mae: best_mae = m; best_w = ws.tolist()

w_knn2, w_knn13, w_rnn, w_hgb, w_mlp = best_w
val_pred = sum(w*p for w,p in zip(best_w, preds))
print(f"\n=== BLEND MAE: {best_mae:.2f} cm ===")
print(f"Weights: {dict(zip(names5, [round(w,3) for w in best_w]))}")

Training HGB (5 seeds)...
Training MLP (α=1.0)...

--- Individual MAEs ---
KNN k=2:      9.49 cm
KNN(1+3)/2:   9.38 cm
RadiusNN 5.9: 9.37 cm
HGB 5-seed:   10.67 cm
MLP α=1.0:    12.31 cm

Optimizing blend...

=== BLEND MAE: 8.79 cm ===
Weights: {'KNN2': 0.0, 'KNN13': 0.392, 'RadNN': 0.254, 'HGB5s': 0.263, 'MLP': 0.091}


In [8]:
# Evaluation
print("=== Validation Set ===")
print_results(y_val, val_pred)

=== Validation Set ===
MAE: 8.792
R2: 90.204


## 7. Predict on Test Set & Save

In [9]:
# ===== Retrain on ALL 3000 + predict test =====
print("Retraining on all 3000...")
t0 = time.time()
X_all_knn = extract_knn_features(images, H, W, C)
X_all_extra = extract_hgb_extra(images, H, W, C)
X_all_hgb = np.hstack([X_all_knn, X_all_extra])

sc_f = StandardScaler(); X_all_s = sc_f.fit_transform(X_all_knn)
sc_hgb_f = StandardScaler(); X_all_hgb_s = sc_hgb_f.fit_transform(X_all_hgb)

# KNN + RadiusNN
knn1_f = KNeighborsRegressor(n_neighbors=1); knn1_f.fit(X_all_s, distances)
knn2_f = KNeighborsRegressor(n_neighbors=2, weights='distance'); knn2_f.fit(X_all_s, distances)
knn3_f = KNeighborsRegressor(n_neighbors=3, weights='distance'); knn3_f.fit(X_all_s, distances)
rnn_f = SafeRadiusNN(radius=5.9, weights='distance', fallback_k=2); rnn_f.fit(X_all_s, distances)

# HGB 5-seed
print("Training HGB (5 seeds)...")
hgb_models_f = []
for seed in hgb_seeds:
    c = HGBClf(n_bins=15, max_iter=500, max_depth=5, learning_rate=0.05,
               min_samples_leaf=15, l2_regularization=0.2, random_state=seed)
    c.fit(X_all_hgb_s, distances); hgb_models_f.append(c)

# MLP α=1.0
print("Training MLP (α=1.0)...")
mlp_f = MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=1000, early_stopping=True,
                     validation_fraction=0.15, learning_rate='adaptive', learning_rate_init=0.001,
                     alpha=1.0, random_state=42)
mlp_f.fit(X_all_s, distances)

# Test predictions
test_raw = np.array(load_test_dataset(config))
X_te_knn = extract_knn_features(test_raw, H, W, C)
X_te_extra = extract_hgb_extra(test_raw, H, W, C)
X_te_s = sc_f.transform(X_te_knn)
X_te_hgb_s = sc_hgb_f.transform(np.hstack([X_te_knn, X_te_extra]))

t_knn2 = knn2_f.predict(X_te_s)
t_knn13 = 0.5 * knn1_f.predict(X_te_s) + 0.5 * knn3_f.predict(X_te_s)
t_rnn = rnn_f.predict(X_te_s)
t_hgb = np.mean([m.predict(X_te_hgb_s) for m in hgb_models_f], axis=0)
t_mlp = mlp_f.predict(X_te_s)

test_pred = w_knn2*t_knn2 + w_knn13*t_knn13 + w_rnn*t_rnn + w_hgb*t_hgb + w_mlp*t_mlp
test_pred = np.clip(test_pred, 0.1, 4.0)
save_results(test_pred)
print(f"\nDone ({time.time()-t0:.1f}s)")
print(f"Saved! Range: {test_pred.min():.3f} - {test_pred.max():.3f} m")
print(f"Weights: KNN2={w_knn2:.3f} KNN13={w_knn13:.3f} RadNN={w_rnn:.3f} HGB={w_hgb:.3f} MLP={w_mlp:.3f}")

Retraining on all 3000...
Training HGB (5 seeds)...
Training MLP (α=1.0)...

Done (630.3s)
Saved! Range: 0.444 - 2.708 m
Weights: KNN2=0.000 KNN13=0.392 RadNN=0.254 HGB=0.263 MLP=0.091
